## Lab 2 &ndash; Hashed structures

Done By, <span style="color:red">
* Téophile Pradalier-Lefebvre</span>
* Nathan Boudry
* Mishael Natth VISWANATHAN
* Aaron Rohit JEROLD

### A) A simple hash function

Python provides its own hash function used for internal purposes:


In [ ]:
hash("Hello world")

5941223518979772549

It is however very prone to collisions, so not well-suited for the task of providing a unique fingerprint for data structures.

In [ ]:
hash(2**64) == hash(8)

True

We'll instead implement today our own toy hash function that transforms a string `S` into an integer:

- initialize `h` to 0
- incorporate successively each character of `S` by
    - adding its `ord` (Unicode codepoint) to `h`
    - scrambling the bits of `h` by XORing it (`^` operator in Python) with 31 times itself
    - replacing `h` by its remainder modulo $2^{32}$ to prevent `h` from growing in size
- return the final value of `h`.
    
Implement this by defining a function `simple_hash()` below.

In [ ]:
def toy_hash(S):
    h = 0

    for character in S:
        h += ord(character)
        h ^= 31 * h
        h %= 2**32

    return h

print(toy_hash("Hello World"))
print(toy_hash("JUNIA"))
print(toy_hash("Bitcoin"))
print(toy_hash("⍺βƔΔε"))

3971264632
2268132318
2028574072
3299468638


In [ ]:
assert toy_hash("JUNIA") == 2268132318

assert toy_hash("Bitcoin") == 2028574072

assert toy_hash("⍺βƔΔε") == 3299468638

'All tests passed successfully!'

'All tests passed successfully!'

### B) Working with blocks

Define a class `Block` that allows to encapsulate some `content` (for example, a list of operations for our blockchain) as well as a list `prev_hashes` of its previous blocks (that may be empty if there is no previous block).

The class `Block` must provide a `.json()` method that returns a string containing a canonical (normalized) description of its members. In other words, we want

- to be able to reconstruct the block from its JSON representation;
- that blocks `B1` and `B2` are to be considered "equal" for all intents and purposes if and only if `B1.json() == B2.json()` (pay attention for example to the order in which the members are written in the string representation).

Finally, `Block`s should have a `.hash()` method that returns the `simple_hash` value of its canonical JSON representation.

In [ ]:
import hashlib
import json

def simple_hash(S):
    return int(hashlib.sha256(bytes(S, "utf8")).digest().hex()[:8], 16)

class Block:
    def __init__(self, content, prev_hashes=None):
        self.content = content
        self.prev_hashes = prev_hashes if prev_hashes is not None else []

    def json(self):
        """Returns a canonical (normalized) JSON string representation of the block."""
        return json.dumps({
            "content": self.content,
            "prev_hashes": self.prev_hashes
        }, sort_keys=True)

    @classmethod
    def from_json(cls, json_str):
        """Reconstructs a Block instance from its JSON representation."""
        data = json.loads(json_str)
        return cls(content=data["content"], prev_hashes=data.get("prev_hashes", []))

    def hash(self):
        """Returns the simple_hash value of its canonical JSON representation."""
        return simple_hash(self.json())

    def __eq__(self, other):
        if isinstance(other, Block):
            return self.json() == other.json()
        return False

    def __repr__(self):
        return f"Block(content={self.content!r}, prev_hashes={self.prev_hashes!r})"

block1 = Block(
    [{"from": "Alice", "to": "Bob", "amount": 5}],
    []
)

block2 = Block(
    [{"from": "Bob", "to": "Alice", "amount": 5}],
    [block1.hash()]
)

print(block1.json())
print(block1.hash())
print(block2.json())
print(block2.hash())

{"content": [{"amount": 5, "from": "Alice", "to": "Bob"}], "prev_hashes": []}
4059570505
{"content": [{"amount": 5, "from": "Bob", "to": "Alice"}], "prev_hashes": [4059570505]}
1426964672


### C) A directed acyclic graph

Consider the following group of blocks (their `content` is just what is written inside).

![DAG.jpg](attachment:DAG.jpg)

([source](363777096_Applying_the_structural_causal_model_framework_for_observational_causal_inference_in_ecology))

Starting with blocks with no successors, add to each block the hashes of its successors (including their own list of hashes). Pay attention to the order in which the blocks need to be visited.

In [ ]:
# Step 1 : Blocks without successors (prev_hashes = [])
carbon_seq = Block("Carbon Sequestration", [])

# Step 2 : Blocks whose successors already have their hash calculated
forest_species = Block("Forest Species Abundance", [carbon_seq.hash()])
fire = Block("Fire", [forest_species.hash()])
poaching = Block("Poaching", [forest_species.hash()])
logging = Block("Logging", [forest_species.hash(), carbon_seq.hash()])
protected_area = Block("Protected Area", [fire.hash(), poaching.hash(), logging.hash()])
# Step 3 : Source blocks
slope = Block("Slope", [protected_area.hash(), logging.hash()])
elevation = Block("Elevation", [protected_area.hash(), logging.hash()])
distance_roads = Block(
    "Distance to Roads and Cities",
    [fire.hash(), protected_area.hash(), poaching.hash(), logging.hash()]
)
#symple hash was too prone to collisions : The code correctly constructs the DAG,
# but the toy hash isn't good enough to produce a unique fingerprint for each block so consequently, multiple blocks share the same key and overwrite each other in the dictionary.

The resulting hashed DAG can be stored as a dictionary of `hash: block` pairs that can be easily checked for coherence (by verifying for each entry that `block.hash() == hash`).

In [ ]:

# 4. Store in a dictionary {hash: block} and test for consistency
all_blocks = [
    carbon_seq, fire, forest_species, poaching,
    logging, protected_area, slope, elevation, distance_roads
]
dag = {b.hash(): b for b in all_blocks}
# Check for consistency as required by the subject
assert all(h == block.hash() for h, block in dag.items())
print("DAG consistency verified successfully! Number of blocks :", len(dag))
print(dag)

DAG consistency verified successfully! Number of blocks : 9
{750427841: Block(content='Carbon Sequestration', prev_hashes=[]), 2401865818: Block(content='Fire', prev_hashes=[999025108]), 999025108: Block(content='Forest Species Abundance', prev_hashes=[750427841]), 1700124450: Block(content='Poaching', prev_hashes=[999025108]), 1253065792: Block(content='Logging', prev_hashes=[999025108, 750427841]), 4102815724: Block(content='Protected Area', prev_hashes=[2401865818, 1700124450, 1253065792]), 2393102687: Block(content='Slope', prev_hashes=[4102815724, 1253065792]), 2634414854: Block(content='Elevation', prev_hashes=[4102815724, 1253065792]), 3645164821: Block(content='Distance to Roads and Cities', prev_hashes=[2401865818, 4102815724, 1700124450, 1253065792])}


### D) Fun with cycles

While not impossible, it is significantly harder to come up with coherent examples of hashed directed graphs in the presence of cycles. Are you able, for example to assign a coherent hash to each block in the following diagram?

![flowchart.png](attachment:flowchart.png)

[source](https://xkcd.com/1195/)

You will most likely need to search by brute force a value that works for the self-referential block. If that turns out to be too long, you can try to make collisions easier to find by reducing the size of the hash.

In [ ]:
def small_hash(S):
    return simple_hash(S) % 2**16


def small_block_hash(block):
    return small_hash(block.json())


for h in range(2**16):
    trap = Block(
        "HEY, WAIT, THIS FLOWCHART IS A TRAP!",
        [h]
    )

    if small_block_hash(trap) == h:
        print("Hash trouvé :", h)
        break


start = Block("START", [h])

print("Hash du TRAP :", small_block_hash(trap))
print("Hash du START :", small_block_hash(start))

Hash du TRAP : 63459
Hash du START : 35346
